# Финальное обучение: SharedDepthUNet на dataset v2

Зафиксированная архитектура v1 обучается на полном train split (`24 000` четвёрок) и проверяется на
полном validation split (`2 000`). Конфигурация следует завершённым screen-тестам:

- shared plane-wise encoder и embeddings расстояний;
- multi-scale fusion четырёх плоскостей;
- фазовая голова `pi * tanh`;
- официальный max-normalized Fresnel reconstruction;
- `loss = (1 - CC) + 1.5 * (1 - SSIM)`;
- без luminance loss, unit-complex, Transformer и итеративного refinement.

Notebook сохраняет `last.pt`, `best_joint.pt`, `best_cc.pt`, `best_ssim.pt`, историю, графики и пример
экспорта после 8-битного квантования ДОЭ. Для продолжения прерванного запуска задайте
`RESUME_CHECKPOINT` в конфигурационной ячейке.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from pytorch_msssim import ssim
except ImportError:
    print('Installing pytorch-msssim...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v2')
OUTPUT_DIR = Path('/kaggle/working/final_shared_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set this to last.pt from an attached Kaggle Dataset to resume across sessions.
RESUME_CHECKPOINT = None

IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)

TRAIN_SEED = 42
MAX_EPOCHS = 30
MIN_EPOCHS = 12
EARLY_STOP_PATIENCE = 6
BATCH_SIZE = 16
LR = 3e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
SSIM_WEIGHT = 1.5
NUM_WORKERS = min(2, os.cpu_count() or 1)
LOG_EVERY = 100

def find_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_root(KAGGLE_DATASET_DIR)
metadata = json.loads((DATA_ROOT / 'dataset_metadata.json').read_text())
if metadata.get('schema_version') != 2:
    raise RuntimeError(f'Expected dataset schema 2, got {metadata.get("schema_version")}')
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'

CONFIG = {
    'architecture': 'SharedDepthUNet',
    'dataset_schema': 2,
    'train_seed': TRAIN_SEED,
    'max_epochs': MAX_EPOCHS,
    'min_epochs': MIN_EPOCHS,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'min_lr': MIN_LR,
    'weight_decay': WEIGHT_DECAY,
    'ssim_weight': SSIM_WEIGHT,
    'image_size': IMAGE_SIZE,
    'wavelength': WAVELENGTH,
    'pixel_pitch': PIXEL_PITCH,
    'zs': ZS,
}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'disabled')

In [ ]:
RESAMPLE = Image.Resampling.LANCZOS

def normalize_image(array: np.ndarray) -> torch.Tensor:
    values = torch.from_numpy(np.asarray(array, dtype=np.float32) / 255.0)
    maximum = values.amax()
    if maximum <= 1e-8:
        raise ValueError('All-black image cannot be normalized')
    return values / maximum

def font_for_size(size: int):
    try:
        return ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError:
        return ImageFont.load_default()

def draw_primitive(mask: Image.Image, rng: np.random.Generator, remaining_fraction: float) -> None:
    draw = ImageDraw.Draw(mask)
    target_area = max(12.0, remaining_fraction * IMAGE_SIZE * IMAGE_SIZE * float(rng.uniform(0.25, 0.65)))
    kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'glyph'], p=[0.28, 0.28, 0.16, 0.14, 0.14]))
    aspect = float(np.exp(rng.uniform(np.log(0.45), np.log(2.2))))
    width = int(np.clip(np.sqrt(target_area * aspect), 3, IMAGE_SIZE - 4))
    height = int(np.clip(np.sqrt(target_area / aspect), 3, IMAGE_SIZE - 4))
    x0 = int(rng.integers(2, max(3, IMAGE_SIZE - width - 1)))
    y0 = int(rng.integers(2, max(3, IMAGE_SIZE - height - 1)))
    x1, y1 = min(IMAGE_SIZE - 2, x0 + width), min(IMAGE_SIZE - 2, y0 + height)

    if kind == 'ellipse':
        draw.ellipse((x0, y0, x1, y1), fill=255)
    elif kind == 'rectangle':
        draw.rectangle((x0, y0, x1, y1), fill=255)
    elif kind == 'line':
        line_width = max(1, int(np.clip(target_area / max(width, height), 1, 10)))
        draw.line((x0, y0, x1, y1), fill=255, width=line_width)
    elif kind == 'ring':
        ring_width = max(1, int(np.clip(min(width, height) * 0.12, 1, 9)))
        draw.ellipse((x0, y0, x1, y1), outline=255, width=ring_width)
    else:
        char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
        font = font_for_size(max(10, min(80, height)))
        glyph = Image.new('L', (max(16, width * 2), max(16, height * 2)), 0)
        gd = ImageDraw.Draw(glyph)
        box = gd.textbbox((0, 0), char, font=font)
        gd.text((-box[0], -box[1]), char, fill=255, font=font)
        box = glyph.getbbox()
        if box is not None:
            glyph = glyph.crop(box).resize((max(3, width), max(3, height)), RESAMPLE)
            mask.paste(glyph, (x0, y0), glyph)

def make_mask(seed: int, target_coverage: float) -> tuple[Image.Image, float]:
    best_mask, best_coverage, best_error = None, 0.0, float('inf')
    for attempt in range(8):
        rng = np.random.default_rng(seed + 1009 * attempt)
        mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
        for _ in range(18):
            coverage = float((np.asarray(mask) > 16).mean())
            error = abs(coverage - target_coverage)
            if error < best_error:
                best_mask, best_coverage, best_error = mask.copy(), coverage, error
            if coverage >= target_coverage * 0.96:
                break
            draw_primitive(mask, rng, max(target_coverage - coverage, 0.002))
        coverage = float((np.asarray(mask) > 16).mean())
        error = abs(coverage - target_coverage)
        if error < best_error:
            best_mask, best_coverage, best_error = mask.copy(), coverage, error
        if target_coverage * 0.75 <= coverage <= target_coverage * 1.25:
            return mask, coverage
    return best_mask, best_coverage

def render_synthetic(item: dict, return_coverage: bool = False):
    seed = int(item['seed'])
    rng = np.random.default_rng(seed)
    target = float(item['target_coverage'])
    mask, coverage = make_mask(seed, target)
    mode = item['background_mode']
    if mode == 'dark':
        background = int(rng.integers(0, 19)); foreground = int(rng.integers(max(80, background + 65), 256))
    elif mode == 'light':
        background = int(rng.integers(175, 246)); foreground = int(rng.integers(0, max(1, background - 65)))
    else:
        background = int(rng.integers(25, 101))
        if rng.random() < 0.7:
            foreground = int(rng.integers(min(255, background + 65), 256))
        else:
            foreground = int(rng.integers(0, max(1, background - 20)))
    mask_values = np.asarray(mask, dtype=np.float32) / 255.0
    image = background + mask_values * (foreground - background)
    image = np.clip(np.rint(image), 0, 255).astype(np.uint8)
    tensor = normalize_image(image)
    return (tensor, coverage) if return_coverage else tensor

def load_natural(item: dict, root: Path = BSDS_ROOT) -> torch.Tensor:
    path = root / item['path']
    with Image.open(path) as source:
        image = source.convert('L')
        if item['transform'] == 'full':
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), method=RESAMPLE, color=fill, centering=(0.5, 0.5))
        else:
            crop = item['crop']; size = int(crop['size']); x, y = int(crop['x']), int(crop['y'])
            image = image.crop((x, y, x + size, y + size)).resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
        if item.get('hflip', False):
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        angle = int(item.get('rotation', 0))
        if angle:
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=fill)
        return normalize_image(np.asarray(image, dtype=np.uint8))

def load_item(item: dict) -> torch.Tensor:
    return render_synthetic(item) if item['source'] == 'synthetic' else load_natural(item)

class QuadDataset(Dataset):
    def __init__(self, manifest: Path):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        quad = torch.stack([load_item(item) for item in self.records[index]['items']]).to(torch.float32)
        if quad.shape != (4, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f'Invalid quad shape: {tuple(quad.shape)}')
        return quad

class QuadSourceDataset(QuadDataset):
    def __getitem__(self, index):
        record = self.records[index]
        quad = torch.stack([load_item(item) for item in record['items']]).to(torch.float32)
        sources = torch.tensor([1 if item['source'] == 'bsds' else 0 for item in record['items']], dtype=torch.long)
        return quad, sources

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class BaselineUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class DepthEmbedding(nn.Module):
    def __init__(self, channels):
        super().__init__(); hidden = max(16, channels // 2)
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, depths):
        return self.net(depths[:, None])[:, :, None, None]

class PlaneFusion(nn.Sequential):
    def __init__(self, channels):
        super().__init__(nn.Conv2d(4 * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())

class SharedDepthUNet(nn.Module):
    def __init__(self):
        super().__init__(); self.planes = 4; channels = (32, 64, 128, 256, 512)
        self.encoders = nn.ModuleList([DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256), DoubleConv(256, 512)])
        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])
        self.fusions = nn.ModuleList([PlaneFusion(c) for c in channels])
        self.pool = nn.MaxPool2d(2)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
        z = torch.tensor(ZS, dtype=torch.float32) * 1e3
        self.register_buffer('depths', (z - z.mean()) / (z.max() - z.min()))
    def forward(self, x):
        batch, planes, height, width = x.shape
        features = x.reshape(batch * planes, 1, height, width); depths = self.depths.repeat(batch); fused = []
        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):
            features = encoder(features) + embedding(depths).to(features.dtype)
            _, channels, h, w = features.shape
            fused.append(fusion(features.reshape(batch, planes * channels, h, w)))
            if level + 1 < len(self.encoders): features = self.pool(features)
        e0, e1, e2, e3, x = fused
        x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(IMAGE_SIZE, dtype=torch.float64) - IMAGE_SIZE / 2) * PIXEL_PITCH
        u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, phase):
        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float()); outputs = []
        for kernel in self.kernels:
            value = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(value)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_loss(target, reconstruction):
    cc = pearson_cc(target, reconstruction)
    sv = torch.stack([
        ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False)
        for m in range(4)
    ], dim=1)
    loss = (1 - cc).mean() + SSIM_WEIGHT * (1 - sv).mean()
    return loss, cc, sv


## Полный train и validation

Checkpoint-стратегия не сворачивает две официальные метрики в единственный результат:

- `best_joint.pt` — минимальный обучающий validation loss;
- `best_cc.pt` — максимальный средний КК;
- `best_ssim.pt` — максимальный средний ИСС;
- `last.pt` — состояние для безопасного продолжения обучения.

Early stopping отслеживает только `best_joint` после минимум 12 эпох. Метрики сохраняются по каждой
плоскости и отдельно для synthetic/BSDS. Порядок train-батчей детерминирован по номеру эпохи, поэтому
resume начинает следующую эпоху с корректным shuffle.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU: Settings -> Accelerator -> GPU')
device = torch.device('cuda')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(TRAIN_SEED)
train_dataset = QuadSourceDataset(DATA_ROOT / 'manifests' / 'train.jsonl')
val_dataset = QuadSourceDataset(DATA_ROOT / 'manifests' / 'val.jsonl')
print('Full splits:', {'train': len(train_dataset), 'val': len(val_dataset)})

def make_train_loader(epoch):
    generator = torch.Generator().manual_seed(TRAIN_SEED + 10_000 + epoch)
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
    )

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

physics = FresnelLayer(device)
model = SharedDepthUNet().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_multiplier(epoch):
    if epoch < 1:
        return epoch + 1
    progress = (epoch - 1) / max(1, MAX_EPOCHS - 1)
    return (MIN_LR / LR) + (1 - MIN_LR / LR) * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
scaler = torch.amp.GradScaler('cuda')

@torch.no_grad()
def validate():
    model.eval()
    loss_total, count = 0.0, 0
    cc_plane = torch.zeros(4, device=device)
    ssim_plane = torch.zeros(4, device=device)
    source_sums = {
        0: {'cc': 0.0, 'ssim': 0.0, 'n': 0},
        1: {'cc': 0.0, 'ssim': 0.0, 'n': 0},
    }
    boundary_sum = 0.0
    phase_pixels = 0

    for target, sources in val_loader:
        target = target.to(device, non_blocking=True)
        sources = sources.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda'):
            phase = math.pi * torch.tanh(model(2 * target - 1))
        reconstruction = physics(phase.float())
        loss, cc, sv = doe_loss(target, reconstruction)
        batch = target.shape[0]
        loss_total += loss.item() * batch
        count += batch
        cc_plane += cc.sum(0)
        ssim_plane += sv.sum(0)
        boundary_sum += float((phase.abs() > 0.95 * math.pi).sum().item())
        phase_pixels += phase.numel()
        for source_id in (0, 1):
            mask = sources == source_id
            n = int(mask.sum().item())
            source_sums[source_id]['cc'] += float(cc[mask].sum().item())
            source_sums[source_id]['ssim'] += float(sv[mask].sum().item())
            source_sums[source_id]['n'] += n

    cc_values = (cc_plane / count).cpu().tolist()
    ssim_values = (ssim_plane / count).cpu().tolist()
    row = {
        'val_loss': loss_total / count,
        'val_cc': float(np.mean(cc_values)),
        'val_ssim': float(np.mean(ssim_values)),
        'val_cc_min': float(np.min(cc_values)),
        'val_ssim_min': float(np.min(ssim_values)),
        'phase_boundary_fraction': boundary_sum / phase_pixels,
    }
    row.update({f'cc_z{m+1}': value for m, value in enumerate(cc_values)})
    row.update({f'ssim_z{m+1}': value for m, value in enumerate(ssim_values)})
    for source_id, label in ((0, 'synthetic'), (1, 'bsds')):
        values = source_sums[source_id]
        row[f'cc_{label}'] = values['cc'] / values['n']
        row[f'ssim_{label}'] = values['ssim'] / values['n']
    return row

def write_history(history):
    if not history:
        return
    with (OUTPUT_DIR / 'metrics.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)
    (OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

def save_checkpoint(path, state):
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(state, temporary)
    temporary.replace(path)

history = []
start_epoch = 0
best = {'joint_loss': float('inf'), 'cc': -float('inf'), 'ssim': -float('inf')}
epochs_without_joint_improvement = 0

if RESUME_CHECKPOINT is not None:
    resume_path = Path(RESUME_CHECKPOINT)
    checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = int(checkpoint['epoch']) + 1
    history = checkpoint.get('history', [])
    best = checkpoint.get('best', best)
    epochs_without_joint_improvement = checkpoint.get('epochs_without_joint_improvement', 0)
    print('Resumed from:', resume_path, 'next epoch:', start_epoch + 1)

for epoch in range(start_epoch, MAX_EPOCHS):
    train_loader = make_train_loader(epoch)
    model.train()
    started = time.perf_counter()
    train_total, seen = 0.0, 0

    for batch_index, (target, _) in enumerate(train_loader, start=1):
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda'):
            phase = math.pi * torch.tanh(model(2 * target - 1))
        with torch.autocast(device_type='cuda', enabled=False):
            loss, _, _ = doe_loss(target, physics(phase.float()))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_total += loss.item() * target.shape[0]
        seen += target.shape[0]

        if batch_index == 1 or batch_index % LOG_EVERY == 0 or batch_index == len(train_loader):
            elapsed = time.perf_counter() - started
            eta = elapsed / batch_index * (len(train_loader) - batch_index)
            print(
                f'epoch {epoch+1}/{MAX_EPOCHS}: batch {batch_index}/{len(train_loader)}, '
                f'loss={loss.item():.4f}, ETA={eta/60:.1f} min',
                flush=True,
            )

    row = validate()
    row.update({
        'epoch': epoch + 1,
        'train_loss': train_total / seen,
        'lr': optimizer.param_groups[0]['lr'],
        'epoch_seconds': time.perf_counter() - started,
    })
    history.append(row)
    scheduler.step()

    improved_joint = row['val_loss'] < best['joint_loss']
    improved_cc = row['val_cc'] > best['cc']
    improved_ssim = row['val_ssim'] > best['ssim']
    if improved_joint:
        best['joint_loss'] = row['val_loss']
        epochs_without_joint_improvement = 0
    else:
        epochs_without_joint_improvement += 1
    if improved_cc:
        best['cc'] = row['val_cc']
    if improved_ssim:
        best['ssim'] = row['val_ssim']

    state = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'history': history,
        'best': best,
        'epochs_without_joint_improvement': epochs_without_joint_improvement,
        'config': CONFIG,
        'parameter_count': parameter_count,
        'metrics_at_save': row,
    }
    save_checkpoint(OUTPUT_DIR / 'last.pt', state)
    if improved_joint:
        save_checkpoint(OUTPUT_DIR / 'best_joint.pt', state)
    if improved_cc:
        save_checkpoint(OUTPUT_DIR / 'best_cc.pt', state)
    if improved_ssim:
        save_checkpoint(OUTPUT_DIR / 'best_ssim.pt', state)
    write_history(history)
    print(row, flush=True)
    print('Best:', best, 'joint patience:', epochs_without_joint_improvement, flush=True)

    del train_loader
    if epoch + 1 >= MIN_EPOCHS and epochs_without_joint_improvement >= EARLY_STOP_PATIENCE:
        print(f'Early stopping after epoch {epoch+1}.', flush=True)
        break

checkpoint_summary = {}
for filename in ('best_joint.pt', 'best_cc.pt', 'best_ssim.pt', 'last.pt'):
    path = OUTPUT_DIR / filename
    if path.exists():
        saved = torch.load(path, map_location='cpu', weights_only=False)
        checkpoint_summary[filename] = {
            'epoch': int(saved['epoch']) + 1,
            'metrics': saved['metrics_at_save'],
        }
(OUTPUT_DIR / 'checkpoint_summary.json').write_text(
    json.dumps(checkpoint_summary, indent=2), encoding='utf-8'
)
print(json.dumps(checkpoint_summary, indent=2))

epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [row['train_loss'] for row in history], label='train')
axes[0].plot(epochs, [row['val_loss'] for row in history], label='validation')
axes[1].plot(epochs, [row['val_cc'] for row in history], label='CC')
axes[2].plot(epochs, [row['val_ssim'] for row in history], label='SSIM')
for axis, title in zip(axes, ['Loss', 'Validation CC', 'Validation SSIM']):
    axis.set_title(title)
    axis.set_xlabel('epoch')
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160)
plt.show()

In [ ]:
# Export is evaluated from the quantized 8-bit DOE, matching the final submission path.
def phase_to_uint8(phase):
    values = ((phase + math.pi) / (2 * math.pi) * 255.0).round().clamp(0, 255)
    return values.to(torch.uint8)

def uint8_to_phase(values):
    return values.to(torch.float32) / 255.0 * (2 * math.pi) - math.pi

def load_model_checkpoint(checkpoint_path):
    candidate = SharedDepthUNet().to(device)
    saved = torch.load(checkpoint_path, map_location=device, weights_only=False)
    candidate.load_state_dict(saved['model'])
    candidate.eval()
    return candidate, saved

@torch.no_grad()
def export_quad(target, checkpoint_path, export_dir, sources=None):
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)
    candidate, saved = load_model_checkpoint(checkpoint_path)
    target = target.to(device).unsqueeze(0) if target.ndim == 3 else target.to(device)
    with torch.autocast(device_type='cuda'):
        continuous_phase = math.pi * torch.tanh(candidate(2 * target - 1))
    phase_u8 = phase_to_uint8(continuous_phase[0, 0]).cpu()
    Image.fromarray(phase_u8.numpy(), mode='L').save(export_dir / 'doe.png')

    # Read the exact stored representation back before physical reconstruction.
    stored_u8 = torch.from_numpy(np.asarray(Image.open(export_dir / 'doe.png').convert('L'), dtype=np.uint8)).to(device)
    stored_phase = uint8_to_phase(stored_u8)[None, None]
    reconstruction = physics(stored_phase).cpu()
    target_cpu = target.cpu()
    cc = pearson_cc(target_cpu, reconstruction)
    sv = torch.stack([
        ssim(target_cpu[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False)
        for m in range(4)
    ], dim=1)

    for plane in range(4):
        target_u8 = (target_cpu[0, plane].clamp(0, 1) * 255).round().to(torch.uint8).numpy()
        recon_u8 = (reconstruction[0, plane].clamp(0, 1) * 255).round().to(torch.uint8).numpy()
        Image.fromarray(target_u8, mode='L').save(export_dir / f'target_z{plane+1}.png')
        Image.fromarray(recon_u8, mode='L').save(export_dir / f'reconstruction_z{plane+1}.png')

    report = {
        'checkpoint': str(checkpoint_path),
        'checkpoint_epoch': int(saved['epoch']) + 1,
        'cc_per_plane': cc[0].tolist(),
        'ssim_per_plane': sv[0].tolist(),
        'cc_mean': float(cc.mean()),
        'ssim_mean': float(sv.mean()),
        'sources': None if sources is None else list(sources),
    }
    (export_dir / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    return report

def load_png_quad(paths):
    if len(paths) != 4:
        raise ValueError('Exactly four PNG paths are required')
    tensors = []
    for path in paths:
        with Image.open(path) as image:
            image = image.convert('L')
            if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                raise ValueError(f'{path}: expected 128x128, got {image.size}')
            tensors.append(normalize_image(np.asarray(image, dtype=np.uint8)))
    return torch.stack(tensors)

# Automatic smoke export from validation. For final inputs call:
# export_quad(load_png_quad([...four paths...]), OUTPUT_DIR/'best_joint.pt', OUTPUT_DIR/'final_export')
validation_target, validation_sources = val_dataset[0]
export_report = export_quad(
    validation_target,
    OUTPUT_DIR / 'best_joint.pt',
    OUTPUT_DIR / 'validation_export',
    sources=['bsds' if int(value) == 1 else 'synthetic' for value in validation_sources],
)
print(json.dumps(export_report, indent=2))